# Experiment 3.4 — Causal Fixed250 Objective Comparison

## Hypothesis
Progressive fixed-duration prefix supervision will improve streaming evidence accumulation, and may improve final balanced accuracy, without using Relative10 or any future-duration normalization.

The primary comparison is parameter-matched **Whole Fixed250 CE vs Causal Fixed250 Prefix CE**. The final non-empty bin is retained even when it is a **partial final bin**; padded samples beyond valid length are masked to zero.

## Validation standard
Support the primary hypothesis only if both conditions hold:
1. Mean paired final test BA gain `Causal250 - Whole250` is positive.
2. Causal250 beats Whole250 in at least 2 of 3 paired seeds.

Streaming BA from 500–1500 ms is secondary. Relative10 is an offline oracle/reference only and is not part of training, inference, or the primary decision rule.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'scripts').is_dir() and (candidate / 'snn').is_dir():
        repo_root = candidate
        break
else:
    raise FileNotFoundError(f'Could not locate writingRing root from {cwd}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from scripts import experiment_3_4_causal_fixed250_objectives as exp34

root = exp34.results_dir(repo_root)
required = {
    'results': root / 'experiment_3_4_results.csv',
    'summary': root / 'experiment_3_4_summary.csv',
    'paired': root / 'experiment_3_4_paired_final.csv',
    'prefix': root / 'experiment_3_4_prefix_summary.csv',
    'conclusion': root / 'experiment_3_4_conclusion.json',
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Run/finalize Experiment 3.4 first:\n' + '\n'.join(missing))


In [ ]:
results = pd.read_csv(required['results'])
summary = pd.read_csv(required['summary'])
paired = pd.read_csv(required['paired'])
prefix = pd.read_csv(required['prefix'])
conclusion = json.loads(required['conclusion'].read_text())
display(summary)
display(paired)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for objective, group in prefix.groupby('objective', sort=False):
    ax.plot(group['observed_ms'], group['mean_ba'], marker='o', label=objective)
ax.set_xlabel('Observed duration (ms)')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Streaming BA vs observed duration')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Conclusion
The finalized decision below follows the preregistered rule. A supported result means causal prefix supervision improves the parameter-matched Whole250 baseline on average and in at least 2/3 seeds. It does **not** imply that Relative10 is causal, nor that a partial final bin is ignored.

In [ ]:
status = 'SUPPORTED' if conclusion['primary_hypothesis_supported'] else 'NOT SUPPORTED'
gain_pp = 100.0 * conclusion['mean_causal_minus_whole_final_ba']
stream_pp = 100.0 * conclusion['mean_streaming_ba_delta_500_to_1500ms']
text = f'''### Final conclusion: **{status}**
- Mean final BA gain, Causal250 - Whole250: **{gain_pp:.2f} pp**
- Positive paired seeds: **{conclusion['positive_final_ba_seeds']}/3**
- Mean streaming BA delta from 500–1500 ms: **{stream_pp:.2f} pp**
- Partial final bin policy: {conclusion['partial_final_bin_policy']}
'''
display(Markdown(text))
